# Model Training and Evaluation

## Overview and Approach
This notebook is dedicated to transforming raw, continuous EEG recordings into a structured mathematical format suitable for Machine Learning classification (Alzheimer's vs. Frontotemporal Dementia vs. Healthy Controls).

While the dataset (OpenNeuro ds004504) has already been robustly cleaned of artifacts by the original authors using ASR and ICA, raw time-series data is highly dimensional, non-stationary, and poorly . 

To suit the data for standard machine learning classifiers this, our strategy relies on **Feature Engineering via Spectral Analysis**. Instead of feeding raw signal amplitudes into our models, we will translate the data from the time domain to the frequency domain to extract neurophysiological biomarkers.

### Pipeline
* **Epoching (Data Windowing):** We divide the continuous EEG into 4-second windows with a 50% overlap. This serves two purposes: it ensures the signal is relatively stable (stationary) within each window, and it artificially increases our sample size, providing the ML models with thousands of short, labeled examples rather than just 88 long ones.
* **Power Spectral Density (PSD):** We use Welch's method to calculate the PSD. This acts as a mathematical equalizer, showing us the power of the brain's electrical activity at different frequencies.
* **Relative Band Power (RBP):** We extract the relative energy of five specific brainwave bands (Delta, Theta, Alpha, Beta, and Gamma). The shift of brain power from faster bands (Alpha/Beta) to slower bands (Delta/Theta) is a strong clinical indicator of neurodegeneration. 
* **Export:** By the end of this notebook, we will have converted massive continuous EEG files into a clean, tabular dataset of localized brainwave features, ready for training robust classification models.

In [1]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn", "joblib"])

0

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold, cross_validate, GridSearchCV, StratifiedKFold        
from sklearn.feature_selection import SelectKBest, f_classif                            
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

PROCESSED_DIR = '../data/processed/'
MODELS_DIR = '../data/processed/models/'
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5  # GroupKFold folds

print('All imports successful.')

All imports successful.


In [3]:
# Load the feature matrix generated in Notebook 01
csv_path = os.path.join(PROCESSED_DIR, 'eeg_rbp_features.csv')
df = pd.read_csv(csv_path)

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['label'].value_counts())
print(f'\nUnique subjects: {df["subject_id"].nunique()}')
df.head()

Dataset shape: (34788, 298)

Class distribution:
label
AD     14514
CN     12012
FTD     8262
Name: count, dtype: int64

Unique subjects: 88


,subject_id,epoch_id,label,Fp1_Delta,Fp1_Delta_entropy,Fp1_Theta,Fp1_Theta_entropy,Fp1_Alpha,Fp1_Alpha_entropy,Fp1_Beta,...,plv_Fp1-F3,plv_Fp2-F4,plv_F3-C3,plv_F4-C4,plv_T3-T5,plv_T4-T6,plv_P3-O1,plv_P4-O2,plv_Fz-Cz,plv_Cz-Pz
0,sub-001,0,AD,0.912041,0.660931,0.030483,0.889226,0.014585,0.894939,0.016458,...,0.945078,0.974351,0.991117,0.970499,0.969612,0.962305,0.984847,0.978018,0.985938,0.983455
1,sub-001,1,AD,0.872221,0.552665,0.059261,0.910196,0.018784,0.956692,0.022035,...,0.963615,0.982072,0.988467,0.980956,0.979092,0.964789,0.985894,0.957888,0.986329,0.981878
2,sub-001,2,AD,0.850522,0.719931,0.067376,0.866101,0.016662,0.921457,0.026871,...,0.950599,0.978892,0.984801,0.978389,0.957061,0.955684,0.939866,0.950704,0.984005,0.969128
3,sub-001,3,AD,0.789663,0.875366,0.106579,0.899696,0.027835,0.964230,0.026968,...,0.865699,0.916756,0.917882,0.924694,0.865740,0.873429,0.875652,0.908626,0.924961,0.868030
4,sub-001,4,AD,0.823902,0.646076,0.083011,0.831100,0.024096,0.941797,0.029004,...,0.923242,0.954258,0.972310,0.965773,0.955878,0.950646,0.974680,0.931400,0.980994,0.951981


In [4]:
# Separate metadata from features and labels
METADATA_COLS = ['subject_id', 'epoch_id', 'label']
FEATURE_COLS  = [c for c in df.columns if c not in METADATA_COLS]

X      = df[FEATURE_COLS].values          # Feature matrix  (n_epochs, n_features)
groups = df['subject_id'].values          # Group vector    (n_epochs,)
y_raw  = df['label'].values               # String labels   (n_epochs,)

print(f'Feature matrix shape : {X.shape}')
print(f'Number of features   : {len(FEATURE_COLS)}')
print(f'Feature columns (first 10): {FEATURE_COLS[:10]}')

Feature matrix shape : (34788, 295)
Number of features   : 295
Feature columns (first 10): ['Fp1_Delta', 'Fp1_Delta_entropy', 'Fp1_Theta', 'Fp1_Theta_entropy', 'Fp1_Alpha', 'Fp1_Alpha_entropy', 'Fp1_Beta', 'Fp1_Beta_entropy', 'Fp1_Gamma', 'Fp1_Gamma_entropy']


In [5]:
# Encode categorical labels ('AD', 'FTD', 'CN') → integers
le = LabelEncoder()
y  = le.fit_transform(y_raw)

print('Label encoding mapping:')
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls} → {idx}')

# Note: StandardScaler is applied inside each cross-validation fold
# (via sklearn Pipeline) to avoid data leakage from the test fold.
print('\nStandardScaler will be applied inside the CV pipeline to prevent leakage.')

Label encoding mapping:
  AD → 0
  CN → 1
  FTD → 2

StandardScaler will be applied inside the CV pipeline to prevent leakage.


In [6]:
# Configure subject-wise GroupKFold cross-validation
gkf = GroupKFold(n_splits=N_SPLITS)

print(f'GroupKFold with {N_SPLITS} folds')
print(f'Total subjects: {len(np.unique(groups))}')
print(f'~{len(np.unique(groups)) // N_SPLITS} subjects held out per fold')

# Quick sanity check — verify no subject appears in both train and test
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    train_subs = set(groups[train_idx])
    test_subs  = set(groups[test_idx])
    assert len(train_subs & test_subs) == 0, f'Data leakage in fold {fold}!'
    print(f'  Fold {fold+1}: train={len(train_idx)} epochs ({len(train_subs)} subjects) | '
          f'test={len(test_idx)} epochs ({len(test_subs)} subjects)')

print('\n✓ No data leakage detected across all folds.')

GroupKFold with 5 folds
Total subjects: 88
~17 subjects held out per fold
  Fold 1: train=27956 epochs (71 subjects) | test=6832 epochs (17 subjects)
  Fold 2: train=27957 epochs (71 subjects) | test=6831 epochs (17 subjects)
  Fold 3: train=27814 epochs (70 subjects) | test=6974 epochs (18 subjects)
  Fold 4: train=27696 epochs (70 subjects) | test=7092 epochs (18 subjects)
  Fold 5: train=27729 epochs (70 subjects) | test=7059 epochs (18 subjects)

✓ No data leakage detected across all folds.


In [7]:
def make_pipeline_with_fs(classifier, k=50):
    """
    Wraps a classifier in a StandardScaler → Feature Selection → Classifier pipeline.
    Feature selection is performed inside each CV fold to prevent leakage.
    """
    return Pipeline([
        ('scaler', StandardScaler()),
        ('select', SelectKBest(score_func=f_classif, k=k)),   # Keep top k features
        ('clf',    classifier)
    ])

# We'll keep a simple pipeline for models that don't need FS (e.g., LDA)
def make_pipeline(classifier):
    """Simple scaler + classifier pipeline."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    classifier)
    ])

# Define models with appropriate pipelines
MODELS = {
    'KNN': make_pipeline_with_fs(KNeighborsClassifier(n_neighbors=5), k=100),
    'Decision Tree': make_pipeline_with_fs(DecisionTreeClassifier(random_state=RANDOM_STATE), k=100),
    'SVM': make_pipeline_with_fs(SVC(kernel='rbf', C=1.0, probability=True, random_state=RANDOM_STATE), k=100),
    'Logistic Regression': make_pipeline_with_fs(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), k=100),
    'LDA': make_pipeline(LinearDiscriminantAnalysis()),
}
SCORING = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

print(f'Models to train: {list(MODELS.keys())}')
print(f'Scoring metrics: {SCORING}')
print('Note: KNN, DT, SVM, and LR include SelectKBest (k=50) inside the pipeline.')

Models to train: ['KNN', 'Decision Tree', 'SVM', 'Logistic Regression', 'LDA']
Scoring metrics: ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
Note: KNN, DT, SVM, and LR include SelectKBest (k=50) inside the pipeline.


In [8]:
# ============================================================
# Hyperparameter grids for Nested Cross-Validation
# ============================================================
# Grids are kept deliberately small to keep runtime manageable.
# Rule of thumb: C values × gamma values × k values × 3 inner folds × 5 outer folds = total SVM fits


PARAM_GRIDS = {
    'KNN': {
        'clf__n_neighbors': [3, 5, 9],
        'clf__weights':     ['uniform', 'distance'],
        'select__k':        [100, 150, 200]           # ← ajustado
    },
    'Decision Tree': {
        'clf__max_depth':        [None, 5, 15],
        'clf__min_samples_split': [2, 10],
        'select__k':             [100, 150, 200]      # ← ajustado
    },
    'SVM': {
        'clf__C':     [0.1, 10],
        'clf__gamma': ['scale'],
        'select__k':  [100]                          # ← ajustado (fixo para acelerar)
    },
    'Logistic Regression': {
        'clf__C':    [0.01, 1, 10],
        'select__k': [100, 150, 200]                 # ← ajustado
    },
    'LDA': {
        'clf__solver':    ['lsqr'],
        'clf__shrinkage': [None, 'auto', 0.5]
    }
}

print('Hyperparameter grids defined for all models.')

Hyperparameter grids defined for all models.


### Nested Cross-Validation: Honest Performance Estimation

Because we now have **295 features** (95 RBP + 10 PLV + 190 advanced features), we must be careful not to overfit. We use **nested cross-validation**:

- **Outer loop (5-fold GroupKFold):** Splits subjects into 5 folds. Each fold is held out once to estimate the final model performance.
- **Inner loop (3-fold GroupKFold):** For each outer training set, we perform a grid search to find the best hyperparameters (including the number of features `k` in `SelectKBest`).

This ensures that hyperparameter tuning **never sees the test subjects**, providing an unbiased estimate of how well the model will generalize to new patients.

> **Important: This is NOT the final model training.**  
> Nested cross‑validation is only a **performance estimation procedure**. The metrics reported here tell us what accuracy to expect when the chosen pipeline is later trained on the full dataset and applied to completely new subjects.  
> After this evaluation, we will **retrain the best pipeline on all available data** and save that final model for deployment (see the last section of this notebook).

> ⚠️ **Computational Cost:** Nested CV trains many models (5 × 3 × number of hyperparameter combinations). Expect longer runtimes (10–30 minutes depending on hardware).

In [11]:
# ============================================================
# Hyperparameter grids for Nested Cross-Validation
# ============================================================
# Grids are deliberately kept small to keep runtime manageable.
# Rule of thumb: C values × gamma values × k values × 3 inner folds × 5 outer folds = total SVM fits

PARAM_GRIDS = {
    'KNN': {
        # Number of neighbors to consider.
        # Odd numbers avoid ties; 3,5,9 cover a reasonable range.
        'clf__n_neighbors': [3, 5, 9],
        # Weighting strategy: 'uniform' treats all neighbors equally,
        # 'distance' gives closer neighbors more influence.
        'clf__weights':     ['uniform', 'distance'],
        # Number of top features to retain (via SelectKBest).
        # Values are chosen relative to our total feature count (up to 295).
        'select__k':        [100, 150, 200]
    },
    'Decision Tree': {
        # Maximum depth of the tree. None lets the tree expand fully;
        # 5 and 15 provide limited depths to prevent overfitting.
        'clf__max_depth':        [None, 5, 15],
        # Minimum number of samples required to split an internal node.
        # Higher values regularize the tree.
        'clf__min_samples_split': [2, 10],
        # Top features to select. We test a range to balance complexity and performance.
        'select__k':             [100, 150, 200]
    },
    'SVM': {
        # Regularization parameter. Smaller C encourages a smoother decision boundary.
        # Testing extremes (0.1 vs 10) covers the practical range efficiently.
        'clf__C':     [0.1, 10],
        # Kernel coefficient for RBF. 'scale' is usually robust for normalized data.
        # We keep it fixed to reduce the search space.
        'clf__gamma': ['scale'],
        # For SVM we fix k to one value (100) to keep total fits low.
        'select__k':  [100]
    },
    'Logistic Regression': {
        # Inverse regularization strength. Smaller values mean stronger regularization.
        'clf__C':    [0.01, 1, 10],
        # Number of features to select. We explore a moderate range.
        'select__k': [100, 150, 200]
    },
    'LDA': {
        # Solver for LDA. 'lsqr' handles shrinkage well and is efficient.
        'clf__solver':    ['lsqr'],
        # Shrinkage parameter to regularize covariance estimation.
        # None = no shrinkage, 'auto' = Ledoit-Wolf, 0.5 = fixed shrinkage.
        'clf__shrinkage': [None, 'auto', 0.5]
    }
}

print('Hyperparameter grids defined for all models.')

Hyperparameter grids defined for all models.


> Important Note on n_jobs: Setting n_jobs=1 in cross_validate while GridSearchCV uses n_jobs=-1 prevents nested parallelism conflicts.

In [ ]:
# ============================================================
# NESTED CROSS-VALIDATION WITH HYPERPARAMETER TUNING
# ============================================================
# This procedure gives an **unbiased estimate** of how well our model will perform
# on completely new subjects. It is NOT the final model training – that happens later.

# Outer CV: estimates final performance on unseen subjects
# We MUST use GroupKFold so that all epochs from a single subject stay together.
# This prevents data leakage where the model "memorizes" individual subjects.
outer_cv = GroupKFold(n_splits=N_SPLITS)

# Inner CV: selects the best hyperparameters (including k for SelectKBest).
# Because the outer loop already guarantees that the inner training set contains
# only independent subjects, we can safely use any CV strategy here without
# leaking subject identity across the inner folds.
# We use GroupKFold again as a conservative choice, but StratifiedKFold would also be valid.
inner_cv = GroupKFold(n_splits=3)

# Dictionary to store cross-validation scores for all models
cv_results = {}

for model_name, pipeline in MODELS.items():
    print(f'\n{"="*60}')
    print(f'Nested CV for {model_name}...')

    param_grid = PARAM_GRIDS.get(model_name, {})

    if param_grid:
        # GridSearchCV searches over the hyperparameter grid using the inner CV.
        # It refits the best model on the full inner training set.
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring='f1_macro',      # Macro F1 is preferred for imbalanced multi-class
            n_jobs=-1,               # Use all available CPU cores
            refit=True,
            verbose=0
        )

        # The outer cross_validate loop evaluates the GridSearchCV object.
        # Important: we set n_jobs=1 here to avoid nested parallelization conflicts
        # when GridSearchCV already uses n_jobs=-1.
        # The `params={'groups': groups}` ensures that the inner GroupKFold receives
        # the correct group labels for the current training split.
        scores = cross_validate(
            grid_search,
            X, y,
            groups=groups,
            cv=outer_cv,
            scoring=SCORING,
            return_train_score=False,
            n_jobs=1,                        # Must be 1 when inner estimator uses parallelism
            params={'groups': groups},       # Pass groups down to inner GroupKFold
            error_score='raise'
        )
    else:
        # For models without hyperparameter tuning (e.g., LDA), just run outer CV.
        scores = cross_validate(
            pipeline,
            X, y,
            groups=groups,
            cv=outer_cv,
            scoring=SCORING,
            n_jobs=-1,
            error_score='raise'
        )

    cv_results[model_name] = scores
    mean_acc = scores['test_accuracy'].mean()
    std_acc  = scores['test_accuracy'].std()
    print(f'Done. Accuracy: {mean_acc:.4f} ± {std_acc:.4f}')

print('\n✓ All models trained with nested cross-validation.')


Nested CV for KNN...


In [ ]:
# Identify best model by F1-Macro
best_model_name = summary_df['F1 Macro'].idxmax()
print(f'Best model by F1-Macro: {best_model_name}')
print(summary_df.loc[best_model_name, display_cols].round(4))

### Interpreting the Baseline Accuracy (~43–52%)

At first glance, an accuracy of ~52% on a 3‑class problem (chance level ≈ 33%) may seem disappointing. However, in the context of **subject‑wise cross‑validation** and clinical EEG data, this is **expected and realistic**.

#### Why aren't we seeing >90% accuracy?

| Common Pitfall (Data Leakage) | Our Rigorous Approach (No Leakage) |
| --- | --- |
| Mixing epochs from the *same subject* in both training and test sets. | Using `GroupKFold` ensures **all epochs of a subject stay entirely in one fold**. |
| The model memorizes subject‑specific scalp patterns, electrode placement, or noise signatures. | The model is forced to learn **disease‑specific biomarkers** that generalize to *unseen brains*. |

#### The "Inter‑Subject Variability" Challenge

Human brains are remarkably diverse. Two patients with the same diagnosis (e.g., Alzheimer's Disease) can exhibit different patterns of cortical atrophy, different skull conductivities, and different baseline EEG rhythms.

- A model trained on `sub‑001` through `sub‑030` is being tested on `sub‑031`—a brain it has **never seen before**.
- Under these strict conditions, an accuracy significantly above chance (e.g., 52% vs. 33%) demonstrates that the model has successfully extracted **true neurophysiological signals** rather than overfitting to individual artifacts.

#### Why This is Good Science

Many published studies report >95% accuracy but fail to separate subjects correctly. Such models are clinically useless because they cannot diagnose a *new* patient walking into the clinic. Our pipeline produces **honest, reproducible, and clinically translatable** performance metrics.

> **Next Step:** To improve these baseline scores, we would typically perform **hyperparameter tuning** *inside* the `GroupKFold` loop (Nested Cross-Validation) and explore more complex feature extraction (e.g., functional connectivity metrics).

In [ ]:
# Plot: Accuracy comparison bar chart with error bars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_sorted = summary_df.sort_values('Accuracy', ascending=False)

# Accuracy
axes[0].bar(
    models_sorted.index,
    models_sorted['Accuracy'],
    yerr=models_sorted['Accuracy Std'],
    capsize=5, color='steelblue', alpha=0.8
)
axes[0].set_title('Mean Accuracy (5-Fold GroupKFold)', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[0].patches, models_sorted['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# F1-Macro
models_f1 = summary_df.sort_values('F1 Macro', ascending=False)
axes[1].bar(
    models_f1.index,
    models_f1['F1 Macro'],
    yerr=models_f1['F1 Macro Std'],
    capsize=5, color='darkorange', alpha=0.8
)
axes[1].set_title('Mean F1-Score Macro (5-Fold GroupKFold)', fontsize=13)
axes[1].set_ylabel('F1 Macro')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(axes[1].patches, models_f1['F1 Macro']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'model_comparison.png'), dpi=150)
plt.show()
print('Plot saved to processed directory.')

### Saving Models and Metadata for Reproducibility

After cross-validation, we **retrain each model on the full dataset** and persist the following artifacts:

| Artifact | Purpose |
| :--- | :--- |
| `*.joblib` (model pipeline) | Contains the trained `StandardScaler` + classifier. Allows us to make predictions on **new, unseen EEG epochs** without re‑running the preprocessing pipeline. |
| `label_encoder.joblib` | Maps string labels (`'AD'`, `'CN'`, `'FTD'`) to integers. Essential for decoding model predictions back to human‑readable class names. |
| `feature_cols.joblib` | Stores the exact column names and order expected by the model. Guarantees that future input data matches the training schema. |
| `cv_summary.csv` | Tabular summary of cross‑validation performance (mean ± std). Useful for quick reporting and model comparison. |
| `cv_results.joblib` | Raw cross‑validation scores for each fold. Enables deeper statistical analysis (e.g., paired t‑tests between models). |

<br>

> **Why retrain on all data?**  
> Cross‑validation estimates how well the model will perform on unseen subjects. Once we trust that estimate, we use **all available labeled epochs** to train the final model—maximizing its exposure to the data before deployment.

In [ ]:
# Retrain ALL models on the complete dataset with hyperparameter tuning (if applicable) and save them for evaluation in Notebook 03

print('\nTraining final models on full dataset with hyperparameter tuning...')

for model_name, pipeline in MODELS.items():
    print(f'  Tuning {model_name}...', end=' ')
    
    param_grid = PARAM_GRIDS.get(model_name, {})
    
    if param_grid:
        # Use GroupKFold on full dataset to find best params
        final_cv = GroupKFold(n_splits=3)
        grid_search = GridSearchCV(
            pipeline, param_grid, cv=final_cv, scoring='accuracy', n_jobs=-1
        )
        grid_search.fit(X, y, groups=groups)
        best_model = grid_search.best_estimator_
        print(f'Best params: {grid_search.best_params_}')
    else:
        best_model = pipeline
        best_model.fit(X, y)
        print('Done.')
    
    safe_name = model_name.lower().replace(' ', '_')
    model_path = os.path.join(MODELS_DIR, f'{safe_name}.joblib')
    joblib.dump(best_model, model_path)
    print(f'    Saved: {model_path}')

# Save encoders and metadata
joblib.dump(le,           os.path.join(MODELS_DIR, 'label_encoder.joblib'))
joblib.dump(FEATURE_COLS, os.path.join(MODELS_DIR, 'feature_cols.joblib'))

summary_df.to_csv(os.path.join(PROCESSED_DIR, 'cv_summary.csv'))
joblib.dump(cv_results, os.path.join(PROCESSED_DIR, 'cv_results.joblib'))

print(f'\n✓ All tuned models and metadata saved to {MODELS_DIR}')
print(f'✓ CV summary saved to {PROCESSED_DIR}cv_summary.csv')
print(f'\n→ Proceed to Notebook 03 for evaluation.')